# Panel data: is public capital productive?

Munnell (1990) fitted a production function to the 48 US states over 1970-1986 and found public capital productive: a 1% increase in it raises output by 0.15%. This workflow replicates her tables, then asks whether the answer survives standard errors that allow for correlation within each state (sections 2 to 8) and estimators that give each state its own level (section 9).

Sections 2 to 8 follow the Next Steps of Model > Linear Regression: every code cell there is the code of the Linear Regression panel, copied as it is, and the text above it names the tab or Next Step it comes from. Sections 1 and 9 are written for this workflow. The data is `produc`, from PollisDatasets: Munnell's output, capital and employment of the 48 states, 1970-1986.

In [ ]:
Pollis.packages("Regress" => "https://github.com/gragusa/Regress.jl", "PollisDatasets", "Plots", "StatsPlots", "Distributions", "CovarianceMatrices" => "https://github.com/Trumpingtons/CovarianceMatrices.jl"; forks = ("CovarianceMatrices",))

## 1. The article: Munnell (1990)

Munnell (1990, pp. 15-18) estimates ln Q = ln MFP + a ln K + b ln L + c ln G + d U% by least squares on the 816 rows, pooled: Q is gross state product, K private capital, L employment, G state and local public capital, U% the unemployment rate. In `produc` they are `gsp`, `pc`, `emp`, `pcap` and `unemp`.

The cells below print every slope, t-statistic (in absolute value, as she prints them), adjusted R2 and standard error of the regression beside her Tables 5, 6 and 7. The intercepts, ln MFP, are left out: they depend on the units of L, and hers are not consistent across the tables (equations 1 and 3 of Table 5 and Table 6 match with employment in persons, Table 7 with millions of persons, the constrained equations with neither).

In [ ]:
using PollisDatasets, LinearAlgebra, Printf, Statistics

"""
	agreement(value, published)

How many decimals `value` shares with `published`, written as in the source: its decimals give the published precision.
"""
function agreement(value, published)
	d = length(split(published, '.')[end])  # decimals as published
	k = findlast(j -> round(value, digits=j) == round(parse(Float64, published), digits=j), 0:d)  # position in 0:d
	k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals"
end

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		d = length(split(published, '.')[end])  # decimals as published
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, agreement(value, published))
	end
end

munnell = dataset("produc");  # 48 states x 17 years, sorted by state, then year
lnQ, lnK, lnL, lnG, U = log.(munnell.gsp), log.(munnell.pc), log.(munnell.emp), log.(munnell.pcap), munnell.unemp;  # Munnell's Q, K, L, G and U%

"""
	leastsquares(y, regressors; rows, offset)

Least squares with an intercept, as Munnell ran it, on the selected `rows`: the estimates, their t-statistics, the adjusted R2 of `y`,
the standard error of the regression and the residuals. A constrained equation regresses `y - offset`, with the coefficient of `offset` fixed at 1.
"""
function leastsquares(y, regressors; rows=trues(length(y)), offset=zeros(length(y)))
	X = hcat(ones(count(rows)), (x[rows] for x in regressors)...)
	b = X \ (y - offset)[rows]; e = (y - offset)[rows] - X * b; n, k = size(X)
	s2 = sum(abs2, e) / (n - k)  # error variance, assuming independent errors
	r2 = 1 - (sum(abs2, e) / (n - k)) / (sum(abs2, y[rows] .- mean(y[rows])) / (n - 1))  # adjusted R2 of y, as Munnell reports it
	(b = b, t = b ./ sqrt.(s2 .* diag(inv(transpose(X) * X))), r2 = r2, se = sqrt(s2), e = e, rows = rows)
end

"""
	compare(title, equation, names, coefficients, t, r2, se)

Print the slopes of `equation`, their t-statistics, the adjusted R2 and the standard error beside Munnell's, written as she prints them.
"""
compare(title, equation, names, coefficients, t, r2, se) = replicate(title, [
	[(name, equation.b[i + 1], p) for (i, (name, p)) in enumerate(zip(names, coefficients))];
	[("t " * name, abs(equation.t[i + 1]), p) for (i, (name, p)) in enumerate(zip(names, t))];
	[("adjusted R2", equation.r2, r2), ("SE", equation.se, se)]])

**Table 5** (p. 16): private capital only (equations 1 and 2), then with public capital (3 to 5); the constrained equations regress ln Q - ln L on ln K - ln L, so that the coefficient of ln L is 1 (a + b = 1), and in equation 5 on ln G - ln L too (a + b + c = 1). In equation 1 the coefficient of U% is printed without its minus sign: .006, with t 4.0. In equation 3 the coefficient of ln G is 0.1550, on the rounding boundary: .16 to 2 decimals, printed .15. Most t-statistics agree to the printed decimal, a few by 0.1.

In [ ]:
eq1 = leastsquares(lnQ, [lnK, lnL, U]);
compare("Table 5, equation 1: no constraint", eq1, ["ln K", "ln L", "U%"], [".36", ".69", ".006"], ["38.0", "82.4", "4.0"], ".992", ".092")
eq2 = leastsquares(lnQ, [lnK - lnL, U]; offset=lnL);
compare("Table 5, equation 2: a + b = 1", eq2, ["ln K - ln L", "U%"], [".30", "-.002"], ["31.9", "1.0"], ".990", ".103")
eq3 = leastsquares(lnQ, [lnK, lnL, lnG, U]);
compare("Table 5, equation 3: no constraint", eq3, ["ln K", "ln L", "ln G", "U%"], [".31", ".59", ".15", "-.007"], ["30.1", "43.2", "9.0", "4.7"], ".993", ".088")
eq4 = leastsquares(lnQ, [lnK - lnL, lnG, U]; offset=lnL);
compare("Table 5, equation 4: a + b = 1", eq4, ["ln K - ln L", "ln G", "U%"], [".34", ".06", "-.007"], ["39.6", "15.9", "4.6"], ".992", ".090")
eq5 = leastsquares(lnQ, [lnK - lnL, lnG - lnL, U]; offset=lnL);
compare("Table 5, equation 5: a + b + c = 1", eq5, ["ln K - ln L", "ln G - ln L", "U%"], [".27", ".08", "-.002"], ["23.3", "4.4", "1.0"], ".990", ".102")

**Table 6** (p. 17): public capital split into highways and streets (H, `hwy`), water and sewer systems (WS, `water`) and other public capital, mostly buildings (O, `util`).

In [ ]:
eq6 = leastsquares(lnQ, [lnK, lnL, log.(munnell.hwy), log.(munnell.water), log.(munnell.util), U]);
compare("Table 6", eq6, ["ln K", "ln L", "ln H", "ln WS", "ln O", "U%"], [".31", ".55", ".06", ".12", ".01", "-.007"], ["28.1", "35.4", "3.8", "9.6", ".7", "5.2"], ".993", ".085")

**Table 7** (p. 18): the four Census regions, made of the divisions in `region`: Northeast 1-2, North Central 3-4, South 5-7, West 8-9 (9, 12, 16 and 11 states).

In [ ]:
regions = [  # name, divisions, Munnell's private-capital-only and with-public-capital equations: coefficients, t, adjusted R2, SE
	("Northeast", [1, 2], ([".11", ".95", "-.01"], ["3.3", "28.9", "3.2"], ".997", ".068"), ([".09", ".90", ".07", "-.01"], ["2.7", "22.2", "2.3", "3.7"], ".997", ".067")),
	("North Central", [3, 4], ([".34", ".72", "-.003"], ["14.2", "41.2", "1.8"], ".998", ".048"), ([".34", ".62", ".12", "-.004"], ["15.1", "22.3", "4.5", "2.6"], ".998", ".046")),
	("South", [5, 6, 7], ([".42", ".62", "-.01"], ["22.4", "30.3", "4.7"], ".983", ".098"), ([".38", ".36", ".36", "-.02"], ["22.8", "12.0", "10.8", "6.8"], ".988", ".082")),
	("West", [8, 9], ([".54", ".58", "-.02"], ["36.9", "51.4", "7.9"], ".997", ".058"), ([".51", ".53", ".08", "-.02"], ["28.0", "28.7", "3.2", "8.4"], ".997", ".056"))];
table7 = Dict{String, Any}();  # the 8 regional equations, for the Durbin-Watson cell
for (region, divisions, private, public) in regions
	rows = munnell.region .∈ Ref(divisions)
	table7[region] = leastsquares(lnQ, [lnK, lnL, U]; rows)
	compare("Table 7, $region ($(count(rows)) rows)", table7[region], ["ln K", "ln L", "U%"], private...)
	table7[region * " + G"] = leastsquares(lnQ, [lnK, lnL, lnG, U]; rows)
	compare("Table 7, $region with public capital", table7[region * " + G"], ["ln K", "ln L", "ln G", "U%"], public...)
end

**Durbin-Watson.** Munnell prints a Durbin-Watson statistic of 1.5 to 2.1 for every equation: no sign of serial correlation. The statistic compares each residual with the one in the row before it, so it depends on the order of the rows, and nothing else in the tables does: the estimates, t-statistics, R2 and standard errors are the same in any order.

Munnell's Durbin-Watson values are reproduced only with the rows ordered by year, then state, evidently the order of her file. In that order the statistic compares neighbouring states in the same year, not the same state in consecutive years. Ordered by state, then year, the time-series Durbin-Watson is 0.06-0.4: strong serial correlation that her t-statistics do not allow for.

The cell prints her value beside the statistic in three orders: year then state (hers), state then year (the order of `produc`), and within each state only, which leaves out the 47 or fewer pairs of rows where one state ends and the next begins.

In [ ]:
durbinwatson(e) = sum(abs2, diff(e)) / sum(abs2, e)

"""
	withinstate(e, state)

The Durbin-Watson statistic from consecutive years of the same state only: residuals `e` sorted by state, then year.
"""
withinstate(e, state) = sum(abs2(e[i] - e[i - 1]) for i in 2:length(e) if state[i] == state[i - 1]) / sum(abs2, e)

@printf("%-28s%10s%14s%14s%14s\n", "Durbin-Watson", "Munnell", "year, state", "state, year", "within state")
for (label, equation, published) in [("Table 5, equation 1", eq1, "2.0"), ("Table 5, equation 2", eq2, "2.1"), ("Table 5, equation 3", eq3, "1.9"),
		("Table 5, equation 4", eq4, "2.0"), ("Table 5, equation 5", eq5, "2.0"), ("Table 6", eq6, "1.9"),
		("Northeast", table7["Northeast"], "1.5"), ("Northeast + G", table7["Northeast + G"], "1.5"),
		("North Central", table7["North Central"], "2.0"), ("North Central + G", table7["North Central + G"], "2.0"),
		("South", table7["South"], "1.7"), ("South + G", table7["South + G"], "1.7"),
		("West", table7["West"], "1.7"), ("West + G", table7["West + G"], "2.0")]
	state, year = munnell.state[equation.rows], munnell.year[equation.rows]
	byyear = durbinwatson(equation.e[sortperm(collect(zip(year, state)))])  # rows ordered by year, then state
	@printf("%-28s%10s%14.4f%14.4f%14.4f  year, state %s\n", label, published, byyear, durbinwatson(equation.e), withinstate(equation.e, state), agreement(byyear, published))
end

**What this means for the answer.** Munnell's t-statistics assume errors independent across years; the within-state Durbin-Watson says they are not, so her standard errors are too small. Sections 2 to 8 refit her equation 3, the same model as Baltagi's pooled OLS (Table 2.6), with standard errors that allow for correlation within each state over time; section 9 goes further and gives each state its own level.

## 2. Fit

The Code Example of the **Clustered** tab: standard errors robust to correlation within each state over time.

In [ ]:
using Regress, PollisDatasets  # least squares with robust standard errors; curated example datasets

# Example data: Munnell (1990), output, capital and employment of 48 US states observed yearly over 1970-1986. Replace it with your own table.
data = dataset("produc");  # one row per state and year; datasetinfo("produc") describes the columns

model = Regress.ols(data, @formula(log(gsp) ~ log(pcap) + log(pc) + log(emp) + unemp), save_cluster=:state) + vcov(CR1(:state));  # production function, cluster-robust by state

display(coeftable(model))  # estimates, standard errors, t-values, p-values and 95% intervals
println("R2 = ", r2(model))  # share of variance explained

## 3. Replicate: published results

Before trusting the numbers, check that the same code reproduces results published by others: each value is shown in full, rounded to the published precision, and beside the published one.

**Replicate** (Linear Regression › Next Steps › Compare): Rerun this code on data with published results: Baltagi (2005, p. 25, Table 2.6) and Petersen's test data (Kellogg web page, published tables).

In [ ]:
using Downloads, LinearAlgebra, Printf

# 1. Baltagi (2005, p. 25, Table 2.6), OLS row: this model on Munnell's data, classical standard errors, published to 3 decimals
baltagi = Dict("log(pcap)" => (0.155, 0.017), "log(pc)" => (0.309, 0.010), "log(emp)" => (0.594, 0.014), "unemp" => (-0.007, 0.001));
X  = modelmatrix(model);
s2 = deviance(model) / dof_residual(model);
classical = sqrt.(s2 .* diag(inv(transpose(X) * X)));  # Baltagi reports classical standard errors
@printf("%-12s%12s%10s%11s%12s%10s%11s\n", "Baltagi", "Coef", "rounded", "published", "SE", "rounded", "published")
for (i, name) in enumerate(coefnames(model))
	haskey(baltagi, name) || continue  # the intercept is not published
	b, se = baltagi[name]
	same = round(coef(model)[i], digits=3) == b && round(classical[i], digits=3) == se
	@printf("%-12s%12.6f%10.3f%11.3f%12.6f%10.3f%11.3f  %s\n", name, coef(model)[i], coef(model)[i], b, classical[i], classical[i], se, same ? "same to 3 decimals" : "DIFFERENT")
end

# 2. Petersen's test data (Kellogg web page, tables of OLS, White and clustered standard errors): 5000 firm-years, published estimates for x to 4 decimals
file = Downloads.download("https://www.kellogg.northwestern.edu/faculty/petersen/htm/papers/se/test_data.txt");
rows = split.(readlines(file));  # firm, year, x, y
test = (firmid = parse.(Int, getindex.(rows, 1)), year = parse.(Int, getindex.(rows, 2)), x = parse.(Float64, getindex.(rows, 3)), y = parse.(Float64, getindex.(rows, 4)));
ols = Regress.ols(test, @formula(y ~ x));  # Regress reports White (HC1) standard errors by default
k   = findfirst(==("x"), coefnames(ols));
Xt  = modelmatrix(ols);
petersen = ["Coefficient"       => (coef(ols)[k], 1.0348),
	"OLS SE"            => (sqrt(deviance(ols) / dof_residual(ols) * inv(transpose(Xt) * Xt)[k, k]), 0.0286),
	"White SE"          => (stderror(ols + vcov(HC1()))[k], 0.0284),
	"Firm-clustered SE" => (stderror(Regress.ols(test, @formula(y ~ x), save_cluster=:firmid) + vcov(CR1(:firmid)))[k], 0.0506),
	"Year-clustered SE" => (stderror(Regress.ols(test, @formula(y ~ x), save_cluster=:year) + vcov(CR1(:year)))[k], 0.0334)];
@printf("\n%-20s%12s%10s%11s\n", "Petersen: x", "Julia", "rounded", "published")
for (label, (value, published)) in petersen
	@printf("%-20s%12.6f%10.4f%11.4f  %s\n", label, value, value, published, round(value, digits=4) == published ? "same to 4 decimals" : "DIFFERENT")
end

## 4. Visualise

**Fitted vs Observed** (Linear Regression › Next Steps › Visualise): Fitted values against the observed response.

In [ ]:
using Plots

yhat = fitted(model);
y    = response(model);  # observed response
scatter(yhat, y, xlabel="Fitted", ylabel="Observed", label=false, alpha=0.5)
plot!(identity, extrema(yhat)..., linestyle=:dash, label="45-degree line")  # perfect-fit reference

**Coefficient Plot** (Linear Regression › Next Steps › Visualise): Estimates with 95% confidence intervals from the selected standard errors.

In [ ]:
using Plots, Distributions

est = coef(model);
hw  = quantile(TDist(dof_residual(model)), 0.975) .* stderror(model);  # 95% half-width
k   = length(est);
scatter(est, 1:k, xerror=hw, yticks=(1:k, coefnames(model)), label=false)
vline!([0], linestyle=:dash, label=false)  # zero-effect reference

## 5. Diagnose

**Residuals vs Fitted** (Linear Regression › Next Steps › Diagnose): Check linearity and constant variance.

In [ ]:
using Plots

scatter(fitted(model), residuals(model), xlabel="Fitted", ylabel="Residual", label=false, alpha=0.5)
hline!([0], linestyle=:dash, label=false)  # a funnel shape suggests heteroskedasticity

**Residual Q-Q Plot** (Linear Regression › Next Steps › Diagnose): Compare residuals with a normal distribution.

In [ ]:
using StatsPlots, Distributions, Statistics

r = residuals(model)
qqplot(Normal(), (r .- mean(r)) ./ std(r), xlabel="Normal quantiles", ylabel="Standardised residual quantiles")  # points near the line = normal errors

**Leverage and Influence** (Linear Regression › Next Steps › Diagnose): Hat-matrix leverage and Cook distance.

In [ ]:
X  = modelmatrix(model);
h  = vec(sum((X / (transpose(X) * X)) .* X, dims=2));  # diagonal of the hat matrix
n, p = size(X);
s2 = deviance(model) / dof_residual(model);  # residual variance
d  = residuals(model) .^ 2 .* h ./ (p .* s2 .* (1 .- h) .^ 2);  # Cook distance of each observation
println("High leverage (h > 2p/n): ", findall(h .> 2p / n))
println("Influential (D > 4/n):    ", findall(d .> 4 / n))

## 6. Compare: the branch

**Clustering Effect** (Linear Regression › Next Steps › Compare): Classical, heteroskedasticity-robust (HC1) and clustered standard errors.

In [ ]:
using Printf, LinearAlgebra

X  = modelmatrix(model);
s2 = deviance(model) / dof_residual(model);
classical = sqrt.(s2 .* diag(inv(transpose(X) * X)));  # assumes iid errors
se = ["Classical" => classical,
	"HC1" => stderror(model + vcov(HC1())),  # robust to heteroskedasticity only
	"Clustered" => stderror(model)];  # also robust to correlation within each state
@printf("%-12s", ""); foreach(p -> @printf("%14s", first(p)), se); println()
for (i, name) in enumerate(coefnames(model))
	@printf("%-12s", name); foreach(p -> @printf("%14.4f", last(p)[i]), se); println()
end

**The branch.** If any clustered standard error is more than 1.5 times the classical one, the errors are strongly correlated within states, and they may be correlated across states in the same year too: switch to the **Driscoll-Kraay** tab and run its Next Step, Cross-Sectional Dependence. Otherwise keep the clustered model.

In [ ]:
switch = any(stderror(model) ./ classical .> 1.5)  # the rule: clustered more than 1.5 times classical

**If `switch`:** the Code Example of the **Driscoll-Kraay** tab, then **Cross-Sectional Dependence** (Linear Regression › Next Steps › Compare): Classical, clustered by state and Driscoll-Kraay standard errors. Steps 7 and 8 then use this model.

In [ ]:
if switch
	using Regress, PollisDatasets  # least squares with robust standard errors; curated example datasets

	# Example data: Munnell (1990), output, capital and employment of 48 US states observed yearly over 1970-1986. Replace it with your own table.
	data = dataset("produc");  # one row per state and year; datasetinfo("produc") describes the columns

	model = Regress.ols(data, @formula(log(gsp) ~ log(pcap) + log(pc) + log(emp) + unemp)) + vcov(DriscollKraay(Bartlett(2), tis=data.year, iis=data.state));  # production function, Driscoll-Kraay with 2 lags (rule of thumb for 17 years)

	display(coeftable(model))  # estimates, standard errors, t-values, p-values and 95% intervals
	println("R2 = ", r2(model))  # share of variance explained

	using Printf, LinearAlgebra

	X  = modelmatrix(model);
	s2 = deviance(model) / dof_residual(model);
	classical = sqrt.(s2 .* diag(inv(transpose(X) * X)));  # assumes iid errors
	clustered = Regress.ols(data, @formula(log(gsp) ~ log(pcap) + log(pc) + log(emp) + unemp), save_cluster=:state) + vcov(CR1(:state));
	se = ["Classical" => classical,
		"Clustered" => stderror(clustered),  # correlation within each state over time
		"Driscoll-Kraay" => stderror(model)];  # also correlation across states in the same year
	@printf("%-12s", ""); foreach(p -> @printf("%14s", first(p)), se); println()
	for (i, name) in enumerate(coefnames(model))
		@printf("%-12s", name); foreach(p -> @printf("%14.4f", last(p)[i]), se); println()
	end
else
	println("Clustered standard errors kept")
end

## 7. Predict

**Confidence Interval** (Linear Regression › Next Steps › Predict): Predicted mean with a 95% confidence interval.

In [ ]:
using Distributions

newdata = map(c -> c[1:5], data);  # first 5 rows; replace with new observations
X0   = modelcols(formula(model).rhs, newdata);  # design matrix of the new rows
yhat = X0 * coef(model);
se   = sqrt.(vec(sum((X0 * vcov(model)) .* X0, dims=2)));  # standard error of the mean
q    = quantile(TDist(dof_residual(model)), 0.975);
println((prediction=yhat, lower=yhat .- q .* se, upper=yhat .+ q .* se))

**Prediction Interval** (Linear Regression › Next Steps › Predict): Range for a new individual observation.

In [ ]:
using Distributions

newdata = map(c -> c[1:5], data);  # first 5 rows; replace with new observations
X0   = modelcols(formula(model).rhs, newdata);  # design matrix of the new rows
yhat = X0 * coef(model);
s2   = deviance(model) / dof_residual(model);  # error variance of one observation (assumes constant variance)
se   = sqrt.(vec(sum((X0 * vcov(model)) .* X0, dims=2)) .+ s2);  # mean uncertainty + noise of one new observation
q    = quantile(TDist(dof_residual(model)), 0.975);
println((prediction=yhat, lower=yhat .- q .* se, upper=yhat .+ q .* se))  # wider than the confidence interval

## 8. Interpret

**Confidence Intervals** (Linear Regression › Next Steps › Interpret): 95% confidence intervals of the coefficients, from the selected standard errors.

In [ ]:
confint(model)  # one row per coefficient: lower and upper bound

**Standardised Coefficients** (Linear Regression › Next Steps › Interpret): Effect of a one-standard-deviation change in each predictor.

In [ ]:
using Statistics

X = modelmatrix(model)[:, 2:end];  # predictors without the intercept
beta_std = coef(model)[2:end] .* vec(std(X; dims=1)) ./ std(response(model));  # standardised effects
println(beta_std)

## 9. Beyond pooled OLS: Baltagi's panel estimators

The model so far pools all 816 rows, as if the 48 states shared one level of output. Baltagi (2005, Chapters 2 and 3) treats the same data as a panel, where each state, and then each year, can have its own level, and reaches a different answer: OLS finds public capital productive, but once the state effects are in, it is not (p. 25).

The cells below follow his treatment and print every number beside the published one (Stata output in Tables 2.7 to 2.10, TSP in Table 3.7). They are written for this workflow, not copied from a panel: the Within and Maximum Likelihood estimators are also the **Panel Fixed-Effects** and **Mixed-Effects** tabs of the Linear Mixed-Effects Models panel, but no panel has the Between and Swamy-Arora estimators yet.

In [ ]:
Pollis.packages("FixedEffectModels", "Econometrics", "MixedModels", "CategoricalArrays", "ForwardDiff")

**Set-up.** Munnell's data with Baltagi's names, his equation (2.42), ln Y = a + b1 ln K1 + b2 ln K2 + b3 ln L + b4 Unemp. `replicate` is defined in section 1.

In [ ]:
using FixedEffectModels, Econometrics, MixedModels, CategoricalArrays, PollisDatasets, Printf, Statistics, LinearAlgebra

munnell = dataset("produc");  # 48 states x 17 years, sorted by state, then year
panel = (lny = log.(munnell.gsp), lnk1 = log.(munnell.pcap), lnk2 = log.(munnell.pc), lnl = log.(munnell.emp), u = munnell.unemp,
	state = categorical(munnell.state), year = categorical(munnell.year));  # Econometrics.jl needs categorical panel and time columns
eq = @formula(lny ~ lnk1 + lnk2 + lnl + u);  # Baltagi (2005, p. 25, equation 2.42)
slopes = ["lnk1", "lnk2", "lnl", "u"];  # ln K1 (public capital), ln K2 (private capital), ln L (labour), Unemp
N, T, K = 48, 17, 4;  # states, years, slopes

### 9.1 Within: one intercept per state

Baltagi (2005, p. 26, Table 2.8). The F test that all 48 state intercepts are equal is his (2.12): if it rejects, pooled OLS leaves out a variable that matters (p. 25).

In [ ]:
within = reg(panel, @formula(lny ~ lnk1 + lnk2 + lnl + u + fe(state)), save=:fe);  # state fixed effects
pooled = reg(panel, eq);  # pooled OLS, classical standard errors
F_states = ((deviance(pooled) - deviance(within)) / (N - 1)) / (deviance(within) / (N * T - N - K));  # (2.12): all state effects equal
sigma_u = std(FixedEffectModels.fe(within).fe_state[1:T:end]);  # spread of the 48 state effects
sigma_e = sqrt(deviance(within) / (N * T - N - K));  # spread within states
replicate("Within (Baltagi 2005, p. 26, Table 2.8)", [
	[(name, coef(within)[i], p) for (i, (name, p)) in enumerate(zip(slopes, ["-.0261493", ".2920067", ".7681595", "-.0052977"]))];
	[("se " * name, stderror(within)[i], p) for (i, (name, p)) in enumerate(zip(slopes, [".0290016", ".0251197", ".0300917", ".0009887"]))];
	[("F(4,764)", within.F, "3064.81"), ("F states(47,764)", F_states, "75.82"),
	 ("sigma_u", sigma_u, ".09057293"), ("sigma_e", sigma_e, ".03813705"), ("rho", sigma_u^2 / (sigma_u^2 + sigma_e^2), ".8494045")]])

### 9.2 Between: the 48 state averages

Baltagi (2005, p. 26, Table 2.7): least squares on each state's averages over the 17 years, the variation the Within estimator throws away.

In [ ]:
between = fit(BetweenEstimator, eq, panel, panel = :state);  # regression on the 48 state means
replicate("Between (Baltagi 2005, p. 26, Table 2.7)", [
	[(name, coef(between)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".1793651", ".3019542", ".5761274", "-.0038903"]))];
	[("se " * name, stderror(between)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".0719719", ".0418215", ".0563746", ".0099084"]))];
	("_cons", coef(between)[1], "1.589444")])

### 9.3 Random effects: Swamy-Arora

Baltagi (2005, p. 27, Table 2.9): feasible GLS, the state effects treated as random, their variance estimated by the Swamy-Arora method (Stata's `xtreg, re`; the SWAR row of Table 2.6).

In [ ]:
swar = fit(RandomEffectsEstimator, eq, panel, panel = :state, time = :year);  # Swamy-Arora variance components
replicate("Swamy-Arora (Baltagi 2005, p. 27, Table 2.9)", [
	[(name, coef(swar)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".0044388", ".3105483", ".7296705", "-.0061725"]))];
	[("se " * name, stderror(swar)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".0234173", ".0198047", ".0249202", ".0009073"]))];
	("_cons", coef(swar)[1], "2.135411")])

### 9.4 Random effects: maximum likelihood

Baltagi (2005, p. 27, Table 2.10): the same model fitted by maximum likelihood (Stata's `xtreg, mle`), here with MixedModels, the state as a random intercept. The chibar2 statistic tests whether the state effects have any variance at all, against pooled OLS.

In [ ]:
mle = fit(MixedModel, @formula(lny ~ lnk1 + lnk2 + lnl + u + (1 | state)), panel; REML=false, progress=false);  # Gaussian likelihood, not REML
s_u, s_e = only(VarCorr(mle).σρ.state.σ), mle.σ;
ll_pooled = -N * T / 2 * (log(2pi * deviance(pooled) / (N * T)) + 1);  # pooled OLS log likelihood
replicate("Maximum likelihood (Baltagi 2005, p. 27, Table 2.10)", [
	[(name, coef(mle)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".0031446", ".309811", ".7313372", "-.0061382"]))];
	[("se " * name, stderror(mle)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, [".0239185", ".020081", ".0256936", ".0009143"]))];
	[("_cons", coef(mle)[1], "2.143865"), ("sigma_u", s_u, ".085162"), ("sigma_e", s_e, ".0380836"), ("rho", s_u^2 / (s_u^2 + s_e^2), ".8333481"),
	 ("log likelihood", loglikelihood(mle), "1401.9041"), ("chibar2(01)", 2 * (loglikelihood(mle) - ll_pooled), "1149.84")]])

**The standard errors differ, and why.** The estimates and the likelihood agree to 5 decimals or more, but the standard errors only to 2 to 5: MixedModels computes them with the variances of the state effects and of the errors held at their estimates, while Stata inverts the observed information of all parameters together, the slopes and the two variances. The cell below writes down the likelihood of the model, takes its Hessian at the MixedModels estimates with ForwardDiff, and inverts it whole: this gives Stata's standard errors. It is a diagnosis, not a validation: it explains the difference, and the published values are the check.

In [ ]:
using ForwardDiff

@assert all(allequal(munnell.state[i:i + T - 1]) for i in 1:T:N * T)  # rows sorted by state, then year
X1 = hcat(ones(N * T), panel.lnk1, panel.lnk2, panel.lnl, panel.u);

"""
	loglik(b, s2_mu, s2_lambda, s2_nu)

Gaussian log likelihood of the balanced error component model of Baltagi (2005, Chapters 2 and 3): state effects with variance `s2_mu`, year effects with
variance `s2_lambda` (zero for the one-way model) and errors with variance `s2_nu`, from the four eigenvalues of the covariance matrix.
"""
function loglik(b, s2_mu, s2_lambda, s2_nu)
	e = reshape(panel.lny - X1 * b, T, N);  # residuals, years x states
	g = mean(e); es = mean(e, dims=1) .- g; et = mean(e, dims=2) .- g;  # grand mean, state and year deviations
	l1, l2, l3, l4 = s2_nu, T * s2_mu + s2_nu, N * s2_lambda + s2_nu, T * s2_mu + N * s2_lambda + s2_nu;
	q = sum(abs2, e .- es .- et .- g) / l1 + T * sum(abs2, es) / l2 + N * sum(abs2, et) / l3 + N * T * g^2 / l4;
	-(N * T * log(2pi) + (N - 1) * (T - 1) * log(l1) + (N - 1) * log(l2) + (T - 1) * log(l3) + log(l4) + q) / 2
end

H = ForwardDiff.hessian(θ -> loglik(θ[1:5], θ[6], 0.0, θ[7]), [coef(mle); s_u^2; s_e^2]);  # slopes and both variances together
println("log likelihood: ", loglik(coef(mle), s_u^2, 0.0, s_e^2), " (MixedModels ", loglikelihood(mle), ")")
replicate("Maximum likelihood, joint standard errors (Baltagi 2005, p. 27, Table 2.10)",
	[("se " * name, sqrt(inv(-H)[i, i]), p) for (i, (name, p)) in enumerate(zip(["_cons"; slopes], [".1376582", ".0239185", ".020081", ".0256936", ".0009143"]))])

### 9.5 Two-way: state and year effects

Baltagi (2005, p. 47, Table 3.7) adds a level for each year, common to all states: the Within estimator with state and year fixed effects, and iterative maximum likelihood (IMLE, computed with TSP), published to 3 decimals. The joint standard errors, as in 9.4, give TSP's.

In [ ]:
within2 = reg(panel, @formula(lny ~ lnk1 + lnk2 + lnl + u + fe(state) + fe(year)));  # state and year fixed effects
mle2 = fit(MixedModel, @formula(lny ~ lnk1 + lnk2 + lnl + u + (1 | state) + (1 | year)), panel; REML=false, progress=false);
s_mu, s_lambda, s_nu = only(VarCorr(mle2).σρ.state.σ), only(VarCorr(mle2).σρ.year.σ), mle2.σ;
H2 = ForwardDiff.hessian(θ -> loglik(θ[1:5], θ[6], θ[7], θ[8]), [coef(mle2); s_mu^2; s_lambda^2; s_nu^2]);
replicate("Two-way Within (Baltagi 2005, p. 47, Table 3.7)", [
	[(name, coef(within2)[i], p) for (i, (name, p)) in enumerate(zip(slopes, ["-0.030", "0.169", "0.769", "-0.004"]))];
	[("se " * name, stderror(within2)[i], p) for (i, (name, p)) in enumerate(zip(slopes, ["0.027", "0.028", "0.028", "0.001"]))]])
replicate("Two-way IMLE (Baltagi 2005, p. 47, Table 3.7)", [
	[(name, coef(mle2)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, ["0.020", "0.250", "0.750", "-0.004"]))];
	[("se " * name, sqrt(inv(-H2)[i + 1, i + 1]), p) for (i, (name, p)) in enumerate(zip(slopes, ["0.024", "0.023", "0.025", "0.001"]))];
	[("se " * name * " (MixedModels)", stderror(mle2)[i + 1], p) for (i, (name, p)) in enumerate(zip(slopes, ["0.024", "0.023", "0.025", "0.001"]))];
	[("sigma_mu", s_mu, "0.091"), ("sigma_lambda", s_lambda, "0.017"), ("sigma_nu", s_nu, "0.035")]])

### 9.6 The answer: is public capital productive?

The coefficient of public capital (ln K1) under each estimator, with its t-value: significant when the states are pooled or averaged, not once each state has its own level (Baltagi 2005, p. 25).

In [ ]:
@printf("%-26s%12s%10s%10s\n", "Public capital (ln K1)", "estimate", "se", "t")
for (label, beta, s) in [("Pooled OLS", coef(pooled)[2], stderror(pooled)[2]), ("Between", coef(between)[2], stderror(between)[2]),
		("Within", coef(within)[1], stderror(within)[1]), ("Swamy-Arora", coef(swar)[2], stderror(swar)[2]),
		("Maximum likelihood", coef(mle)[2], sqrt(inv(-H)[2, 2])), ("Two-way Within", coef(within2)[1], stderror(within2)[1]),
		("Two-way IMLE", coef(mle2)[2], sqrt(inv(-H2)[2, 2]))]
	@printf("%-26s%12.4f%10.4f%10.2f\n", label, beta, s, beta / s)
end

**The branch: fixed or random state effects?** The random-effects estimates are efficient only if the state effects are unrelated to the inputs; the Within estimates are consistent either way. Hausman's test compares the two (Baltagi 2005, Section 4.3). Baltagi leaves it as an exercise for these data (Exercise 4.17), so there is no published value to check it against. If it rejects at 5%, `fixed` is true and the Within estimates are the ones to keep; otherwise Swamy-Arora.

In [ ]:
using Distributions

q = coef(within) - coef(swar)[2:end];  # Within minus Swamy-Arora, slopes only
V = vcov(within) - vcov(swar)[2:end, 2:end];
hausman = transpose(q) * (V \ q);
@printf("Hausman chi2(%d) = %.2f, p-value = %.4f\n", K, hausman, ccdf(Chisq(K), hausman))
fixed = ccdf(Chisq(K), hausman) < 0.05  # the rule: reject random effects at 5%

**Now in Compose › New Methods.** Table 2.6 (p. 25) has two more random-effects estimators, WALHUS (Wallace and Hussain) and AMEMIYA, which differ from Swamy-Arora only in how they estimate the two variances; Table 3.7 has them for the two-way model too, with SWAR, and Table 9.4 a nested model of states within 9 regions. No Julia package has them. The New Methods notebook **WALHUS, AMEMIYA, Swamy-Arora and nested error components** writes them out, tested against Baltagi's Tables 2.1, 2.6, 3.1, 3.7 and 9.4 and cross-checked with R's plm. Open it from **Compose › New Methods**.

## References

- Baltagi, B. H. (2005). *Econometric Analysis of Panel Data*, 3rd ed. Chichester: Wiley.
- Driscoll, J. C. and Kraay, A. C. (1998). Consistent covariance matrix estimation with spatially dependent panel data. *Review of Economics and Statistics* 80(4), 549-560.
- Hausman, J. A. (1978). Specification tests in econometrics. *Econometrica* 46(6), 1251-1271.
- Munnell, A. H. (1990). How does public infrastructure affect regional economic performance? *New England Economic Review*, September/October, 11-32.
- Petersen, M. A. (2009). Estimating standard errors in finance panel data sets: comparing approaches. *Review of Financial Studies* 22(1), 435-480. Test data and published results: https://www.kellogg.northwestern.edu/faculty/petersen/htm/papers/se/se_programming.htm